# Exploratory Data Analysis (EDA)
## Industrial Equipment Success Score Predictor

This notebook explores the synthetic equipment dataset to understand:
- Feature distributions
- Correlations with the target (Success Score)
- Categorical breakdowns
- Data quality checks

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.generator import generate_equipment_data
from src.config import get_config

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Generate or load data
config = get_config()
df = generate_equipment_data(n_samples=config.data.n_samples, seed=config.project.random_seed)
print(f"Dataset shape: {df.shape}")
df.head()

## 1. Dataset Overview & Summary Statistics

In [ ]:
# Basic info
df.info()
print("\n" + "="*50)

# Numeric summary
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols].describe().round(2)

## 2. Target Variable Distribution (Success Score)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(df['success_score'], kde=True, bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Success Score Distribution')
axes[0].set_xlabel('Success Score')
axes[0].axvline(df['success_score'].mean(), color='red', linestyle='--', label=f'Mean: {df["success_score"].mean():.1f}')
axes[0].legend()

# Box plot
sns.boxplot(y=df['success_score'], ax=axes[1], color='lightcoral')
axes[1].set_title('Success Score Box Plot')

plt.tight_layout()
plt.show()

print(f"Mean: {df['success_score'].mean():.2f}")
print(f"Std:  {df['success_score'].std():.2f}")
print(f"Min:  {df['success_score'].min():.2f}")
print(f"Max:  {df['success_score'].max():.2f}")

## 3. Feature Distributions

In [ ]:
numeric_features = [
    'operating_temperature', 'vibration_level', 'pressure_reading',
    'power_consumption', 'runtime_hours', 'days_since_maintenance',
    'error_count_24h', 'oil_quality_index', 'load_factor', 'ambient_temperature'
]

fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    sns.histplot(df[col], kde=True, bins=40, ax=axes[i], color='teal')
    axes[i].set_title(f'{col} Distribution')
    axes[i].set_xlabel(col)

plt.tight_layout()
plt.show()

## 4. Correlation Matrix

In [ ]:
# Compute correlation matrix for numeric columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with target
target_corr = corr['success_score'].drop('success_score').sort_values(key=abs, ascending=False)
print("Top correlations with Success Score:")
print(target_corr)

## 5. Categorical Feature Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

categoricals = ['equipment_type', 'manufacturer', 'facility_location']

for i, col in enumerate(categoricals):
    sns.boxplot(data=df, x=col, y='success_score', ax=axes[i], palette='Set2')
    axes[i].set_title(f'Success Score by {col.replace("_", " ").title()}')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Count plots
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for i, col in enumerate(categoricals):
    sns.countplot(data=df, x=col, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col.replace("_", " ").title()} Count')
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 6. Feature vs Target Scatter Plots

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    sns.scatterplot(data=df, x=col, y='success_score', ax=axes[i], alpha=0.3, edgecolor=None)
    axes[i].set_title(f'{col} vs Success Score')
    # Add regression line
    sns.regplot(data=df, x=col, y='success_score', ax=axes[i], scatter=False, color='red')

plt.tight_layout()
plt.show()

## 7. Data Quality Checks

In [ ]:
# Missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

# Duplicates
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

# Outliers (IQR method)
print("\nOutlier counts (beyond 1.5*IQR):")
for col in numeric_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
    print(f"  {col}: {outliers} ({100*outliers/len(df):.1f}%)")

## 8. Summary & Insights

### Key Findings:
- **Success Score** is roughly normally distributed with some left skew
- **Vibration level** and **Error count** show strong negative correlation with score
- **Oil quality index** shows strong positive correlation
- Equipment **type** and **manufacturer** create meaningful score differences
- No missing values or duplicates in synthetic data
- Outliers present in power_consumption and runtime_hours (expected in industrial data)